Task 3 – Time Series Analysis of Installs

## Objective

To analyze the trend of total installs over time across selected app categories.

## Dataset

Google Play Store Dataset

## Transformations Applied

* Filtered reviews greater than 500
* Excluded app names containing the letter S
* Excluded app names starting with X, Y, or Z
* Selected categories beginning with E, C, or B
* Applied category translations

## KPI Measured

* Monthly Total Installs
* Month-over-Month Growth Rate

## Visualization

Multi-category Time Series Line Chart with highlighted periods of significant growth (>20% MoM).

## Time Restriction

The chart is available only between 6 PM IST and 9 PM IST.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [2]:
apps_df = pd.read_csv(
    r"C:\Users\HP\Downloads\archive\googleplaystore.csv"
)

reviews_df = pd.read_csv(
    r"C:\Users\HP\Downloads\archive\googleplaystore_user_reviews.csv"
)

In [3]:
reviews_df = reviews_df[
    ['App', 'Sentiment_Subjectivity']
]

reviews_df['Sentiment_Subjectivity'] = pd.to_numeric(
    reviews_df['Sentiment_Subjectivity'],
    errors='coerce'
)

In [4]:
merged_df = apps_df.merge(
    reviews_df,
    on='App',
    how='left'
)

merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Sentiment_Subjectivity
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,NaN
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,1.000000
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0.833333
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0.000000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,NaN


In [5]:
merged_df['Installs'] = (
    merged_df['Installs']
    .str.replace(',', '')
    .str.replace('+', '')
)

merged_df['Installs'] = pd.to_numeric(
    merged_df['Installs'],
    errors='coerce'
)

In [6]:
merged_df['Reviews'] = pd.to_numeric(
    merged_df['Reviews'],
    errors='coerce'
)

In [7]:
merged_df['Last Updated'] = pd.to_datetime(
    merged_df['Last Updated'],
    errors='coerce'
)

merged_df['Month'] = (
    merged_df['Last Updated']
    .dt.to_period('M')
    .astype(str)
)

In [8]:
filtered_df = merged_df[
    (merged_df['Reviews'] > 500) &
    (~merged_df['App'].str.contains('S', case=False, na=False)) &
    (~merged_df['App'].str.startswith(('X','Y','Z'), na=False)) &
    (
        merged_df['Category'].str.startswith('E') |
        merged_df['Category'].str.startswith('C') |
        merged_df['Category'].str.startswith('B')
    )
]

filtered_df.shape

(4853, 15)

In [9]:
filtered_df['Category'].value_counts()

Category
COMMUNICATION          1512
ENTERTAINMENT          1478
BOOKS_AND_REFERENCE     729
BUSINESS                501
EDUCATION               462
COMICS                   85
EVENTS                   82
BEAUTY                    4
Name: count, dtype: int64

In [10]:
category_translation = {
    'BEAUTY': 'Saundarya',          # Hindi
    'BUSINESS': 'Vanigam'         # Tamil
}

filtered_df['Category_Display'] = (
    filtered_df['Category']
    .replace(category_translation)
)

In [11]:
time_df = (
    filtered_df
    .groupby(['Month', 'Category_Display'])['Installs']
    .sum()
    .unstack(fill_value=0)
)

time_df.head()

Category_Display,BOOKS_AND_REFERENCE,COMICS,COMMUNICATION,EDUCATION,ENTERTAINMENT,EVENTS,Saundarya,Vanigam
Month,,,,,,,,
2014-01,0.0,0.0,100000.0,0.0,0.0,0.0,0.0,0.0
2014-03,0.0,0.0,100000.0,0.0,0.0,0.0,0.0,0.0
2014-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100000.0
2014-07,0.0,0.0,400000000.0,10000.0,0.0,0.0,0.0,0.0
2014-10,500000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
growth = time_df.sum(axis=1).pct_change()

growth[growth > 0.20]

Month
2014-07    3999.100000
2014-11       9.000000
2015-06      39.000000
2015-07       9.000000
2016-03      99.000000
2016-04       9.000000
2016-06       3.006000
2016-08      35.363636
2016-09     109.250000
2017-04      10.000000
2017-08    4399.100000
2017-11       6.500000
2018-01     612.846538
2018-03    3609.105000
2018-05     124.951220
2018-06      12.365608
2018-07      11.549733
2018-08       4.900167
dtype: float64

In [15]:
from datetime import datetime
import pytz
import matplotlib.pyplot as plt

ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)

if 18 <= current_time.hour < 21:

    plt.figure(figsize=(14,7))

    # Plot line for each category
    for col in time_df.columns:
        plt.plot(
            time_df.index,
            time_df[col],
            marker='o',
            linewidth=2,
            label=col
        )

    # Highlight growth periods (>20%)
    growth_months = growth[growth > 0.20].index

    for month in growth_months:
        plt.axvspan(
            month,
            month,
            alpha=0.15,
            color='yellow'
        )

    plt.title(
        "Trend of Total Installs Over Time by App Category",
        fontsize=14
    )

    plt.xlabel("Month")
    plt.ylabel("Total Installs (Log Scale)")

    # Makes chart readable
    plt.yscale('log')

    plt.xticks(rotation=45)

    plt.legend(
        title="Category",
        bbox_to_anchor=(1.02, 1),
        loc='upper left'
    )

    plt.grid(True, alpha=0.3)

    plt.tight_layout()

    plt.show()

else:
    print(
        "Line chart is available only between 6 PM IST and 9 PM IST."
    )

Line chart is available only between 6 PM IST and 9 PM IST.
